# Solution B1 · KNN on mixed data: which columns to put in the distance

**Type**: Implement with scikit-learn

## Objective

Build a KNN on mixed data (scaled numerics + one-hot categoricals) with a `ColumnTransformer`, measure in cross-validation what happens when adding an **informative** category (`contract_type`) and a **noisy** one
(`city`), choose the hyperparameters with `GridSearchCV` and compare the result with a logistic regression.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
NUM = ["ratio", "age", "job_tenure", "income", "amount"]
X, y = df[NUM + ["contract_type", "city"]], df["defaulted"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Commented solution

In [ ]:
def build(numerics, categoricals, k=25):
    blocks = []
    if numerics:
        blocks.append(("num", StandardScaler(), numerics))
    if categoricals:
        blocks.append(("cat", OneHotEncoder(handle_unknown="ignore"), categoricals))
    return make_pipeline(ColumnTransformer(blocks, sparse_threshold=0), KNeighborsClassifier(k, weights="distance"))

In [ ]:
variants = {"numerics": [], "+ contract": ["contract_type"], "+ city": ["city"], "+ both": ["contract_type", "city"]}
cv_auc = pd.Series({name: cross_val_score(build(NUM, cat), X_tr, y_tr, cv=cv, scoring="roc_auc").mean() for name, cat in variants.items()})
delta = {"contract": cv_auc["+ contract"] - cv_auc["numerics"], "city": cv_auc["+ city"] - cv_auc["numerics"]}
harmful_variable = min(delta, key=delta.get)
city_loss = cv_auc["numerics"] - cv_auc["+ city"]
print(cv_auc.round(4).to_string())
print(f"change of AUC by adding contract_type: {delta['contract']:+.4f} | by adding city: {delta['city']:+.4f} -> harmful: {harmful_variable}")

In [ ]:
grid = {"kneighborsclassifier__n_neighbors": [5, 15, 25, 45, 75, 125], "kneighborsclassifier__weights": ["uniform", "distance"], "kneighborsclassifier__p": [1, 2]}
search = GridSearchCV(build(NUM, []), grid, cv=cv, scoring="roc_auc").fit(X_tr, y_tr)
best_params = {k.replace("kneighborsclassifier__", ""): v for k, v in search.best_params_.items()}
knn_test_auc = roc_auc_score(y_te, search.predict_proba(X_te)[:, 1])
lr = make_pipeline(ColumnTransformer([("num", StandardScaler(), NUM), ("cat", OneHotEncoder(handle_unknown="ignore"), ["contract_type"])], sparse_threshold=0), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
lr_test_auc = roc_auc_score(y_te, lr.predict_proba(X_te)[:, 1])
print(f"best parameters: {best_params} (cv AUC {search.best_score_:.4f})")
print(f"test AUC: optimized KNN {knn_test_auc:.4f} | logistic {lr_test_auc:.4f}")

## Check

In [ ]:
with verify():
    check(isinstance(build(NUM, ["city"]), Pipeline) and build(NUM, []).fit(X_tr, y_tr).predict(X_te).shape == (len(X_te),), "build returns a Pipeline that also works without categoricals", "the 'cat' block must not be created if the list is empty")
    check(list(cv_auc.index) == ["numerics", "+ contract", "+ city", "+ both"] and 0.72 < cv_auc["numerics"] < 0.82, "four variants; AUC of the numerics between 0.72 and 0.82", "index: numerics, + contract, + city, + both")
    check(harmful_variable == "city" and close(city_loss, cv_auc["numerics"] - cv_auc["+ city"]) and city_loss > 0.005, "the city (noise) lowers the AUC by over 0.005", "compare + contract and + city")
    check(abs(cv_auc["+ contract"] - cv_auc["numerics"]) < 0.02 and cv_auc["+ both"] < cv_auc["+ contract"], "the contract does not help the KNN (difference below 0.02) and adding the city too makes it worse", "read the series cv_auc")
    check(set(best_params) == {"n_neighbors", "weights", "p"} and best_params["n_neighbors"] >= 25, "the search prefers a large k (25 or more): the signal is weak and noisy", "keys without the prefix 'kneighborsclassifier__'")
    check(knn_test_auc > 0.7 and lr_test_auc > knn_test_auc - 0.005, "the logistic is not beaten by the optimized KNN (or nearly tied)", "compare knn_test_auc and lr_test_auc on the test set")
done()

## What you learned

- **Noisy features cost**: adding `city` (12 dummies unrelated to the outcome) lowers the AUC of the KNN, while in a logistic the same variable is simply ignored (coefficients close to 0). In KNN every column
  enters the distance with the same weight, the information is not enough to compensate.
- Even an **informative** category like `contract_type` does not help the KNN here: the distance between two 0/1 dummies is 0 or 1.4, an enormous difference compared to the standardized numerics, and the signal of `unemployed` (7% of the cases) gets lost in the noise of
  all the other coordinates. A logistic uses the same signal well (M4 B1).
- The optimized KNN chooses a large `k`: when the signal is weak and noisy you need many neighbors to average (with small `k` the estimate is too variable). Note that the best (`k = 125`) is **the edge of the grid**: it is the signal to extend it and retry with even larger `k`.
- Honest comparison: the optimized KNN (0.776) and the logistic (0.776) tie. A simpler model, faster at prediction and more interpretable wins at equal AUC: it is the default choice until KNN proves otherwise.

## Common pitfall

Treating dummies as if they were "free" in the distance. A category with 12 levels adds 12 0/1 columns that dominate the standardized numerics when the levels differ: a KNN on data with many categories requires
explicit choices (weight of the block, Gower distance) or a different representation (target encoding, embedding).

## Interview question

<details><summary>How do you treat categorical variables in a distance-based model?</summary>

One-hot with `handle_unknown` is the starting point, but with many categories or few informative ones it adds noise to the distance: I check with an ablation in cross-validation, consider weighting the block (multiplying the dummies by a factor),
reducing the levels or using a distance designed for mixed data (Gower). If the categories matter a lot, a tree model handles them better and does not require these choices.

</details>